In [1]:
import numpy as np
import scipy as sc
import pandas as pd
import pyspark.sql.functions as f
from pyspark.sql import DataFrame, Window

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.datasource.gnomad.ld import GnomADLDMatrix
from gentropy.method.susie_inf import SUSIE_inf
from scipy.stats import chi2
from gentropy.common.spark_helpers import neglog_pvalue_to_mantissa_and_exponent
from gentropy.susie_finemapper import SusieFineMapperStep
from pyspark.sql.functions import explode, col, when
from pyspark.sql.functions import array_contains

from gentropy.finemapping_simulations import FineMappingSimulations

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

Loading BokehJS ...

In [2]:
session = Session()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/05/16 16:28:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/05/16 16:28:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
ld_matrix = np.load('/Users/yt4/Projects/ot_data/tmp/ld_matrix.npy')

ld_index=session.spark.read.parquet("/Users/yt4/Projects/ot_data/tmp/ld_index")

In [4]:
ld_matrix_for_sim=ld_matrix[0:1000,:][:,0:1000]
ld_index_for_sim=ld_index.limit(1000)

x2=FineMappingSimulations.SimulationLoop(
        n_iter=2,
        n_causal=3,
        session=session,
        he2_reggen=0.002,
        sample_size=100_000,
        ld_matrix_for_sim=ld_matrix_for_sim,
        ld_index=ld_index_for_sim
);

24/05/15 15:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/05/15 15:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/05/15 15:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/05/15 15:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/05/15 15:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/05/15 15:03:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/05/15 1

In [5]:
FineMappingSimulations.ProvideSummary(cred_sets=x2,n_causal=3)


{'successful_runs': 2,
 'number_of_cs': 3,
 'expected_results': 6,
 'accuracy': 1.0,
 'accuracy_lead': 1.0,
 'sensitivity': 0.5}

In [6]:
x2

,credibleSetIndex,studyLocusId,studyId,region,exploded_locus,variantId,chromosome,position,credibleSetlog10BF,purityMeanR2,purityMinR2,zScore,pValueMantissa,pValueExponent,is_in_X,is_in_lead
0,2,-1280585399513307521,sim0,"7_100077935_A_G,7_100502552_T_TA,7_100096802_C...",[7_100096802_C_CA],7_100096802_C_CA,7,100096802,12.152110,1.0,1.0,-8.589062,1.140481,-18,True,1
1,1,-26256609997317339,sim0,"7_100077935_A_G,7_100502552_T_TA,7_100096802_C...",[7_100077935_A_G],7_100077935_A_G,7,100077935,38.867161,1.0,1.0,-14.040177,1.130056,-45,True,1
0,1,9077662526290999120,sim1,"7_100520740_G_GTATA,7_100066574_C_CA,7_9996392...",[7_100066574_C_CA],7_100066574_C_CA,7,100066574,13.745062,1.0,1.0,-9.109082,1.203635,-20,True,1


In [7]:
ld_matrix_for_sim=ld_matrix[0:500,:][:,0:500]
ld_index_for_sim=ld_index.limit(500)

In [6]:
from typing import Any
def SimSumStatFromLD(
    n_causal: int,
    he2_reggen: float,
    U: np.ndarray,
    n: int,
    noise: bool = False,
    scale_noise: float = 1,
) -> dict[str, Any]:
    """Simulates summary statistics for a genetic association study.

    Args:
        n_causal (int): number of causal snps.
        he2_reggen (float): Heritability explained by the combined effect of the region and gene.
        U (np.ndarray): LD.
        n (int): Sample size.
        noise (bool, optional): Add noise to the simulation. Defaults to False.
        scale_noise (float, optional): Scale of the noise. Defaults to 1.

    Returns:
        dict[str, Any]: Dictionary containing the simulated summary statistics.
    """
    # Calculate the total number of SNPs in analysis
    M = U.shape[0]

    # Calculate heritability explained by one causal SNP
    Tau = n * he2_reggen / n_causal

    # Simulate the causal status of SNPs
    indexes = np.random.choice(np.arange(M), size=n_causal, replace=False)
    cc = np.repeat(0, M)
    cc[indexes] = 1

    # Simulate joint z-statistics
    jz = np.zeros(M)
    x = np.random.normal(loc=0, scale=1, size=n_causal)
    jz[cc == 1] = x * np.sqrt(Tau)

    # Simulate GWAS z-statistics
    muz = U @ jz
    GWASz = np.random.multivariate_normal(mean=muz, cov=U, size=1)

    GWASz = GWASz.flatten()

    if noise:
        # M1 = int(M * prop_of_snps_to_noise)
        # indexes_causal = indexes[
        #    np.random.choice(np.arange(len(indexes)), size=1, replace=False)
        # ]
        # indexes_noise = np.random.choice(np.arange(M), size=M1, replace=False)
        # combined = np.concatenate((indexes_causal, indexes_noise))
        # unique_elements = np.unique(combined)
        # GWASz[unique_elements] = GWASz[unique_elements] + np.random.normal(
        #    loc=0, scale=scale_noise, size=len(unique_elements)
        # )
        indexes_causal = indexes[
            np.random.choice(np.arange(len(indexes)), size=1, replace=False)
        ]
        x_tmp = U[indexes_causal, :]
        x_tmp = np.abs(x_tmp)
        x_tmp[indexes_causal] = 0
        ind_tmp = np.where(x_tmp > 0.5)
        if len(ind_tmp) >= 2:
            indexes_noise = ind_tmp[
                np.random.choice(np.arange(len(ind_tmp)), size=2, replace=False)
            ]
        else:
            indexes_noise = np.random.choice(M, size=2, replace=False)
        GWASz[indexes_noise] = GWASz[indexes_noise] + np.random.normal(
            loc=0, scale=scale_noise, size=len(indexes_noise)
        )

    GWASp = chi2.sf(GWASz**2, df=1)  # convert Z to Pval

    return {
        "Z": GWASz.flatten(),
        "P": GWASp.flatten(),
        "Tau": Tau,
        "indexes": indexes,
    }


In [9]:
n_causal=1
he2_reggen=0.002
U=ld_matrix_for_sim
n=100_000
noise=True
scale_noise=1

# Calculate the total number of SNPs in analysis
M = U.shape[0]

# Calculate heritability explained by one causal SNP
Tau = n * he2_reggen / n_causal

# Simulate the causal status of SNPs
indexes = np.random.choice(np.arange(M), size=n_causal, replace=False)
cc = np.repeat(0, M)
cc[indexes] = 1

# Simulate joint z-statistics
jz = np.zeros(M)
x = np.random.normal(loc=0, scale=1, size=n_causal)
jz[cc == 1] = x * np.sqrt(Tau)

# Simulate GWAS z-statistics
muz = U @ jz
GWASz = np.random.multivariate_normal(mean=muz, cov=U, size=1)

GWASz = GWASz.flatten()

In [12]:
indexes_causal = indexes[
    np.random.choice(np.arange(len(indexes)), size=1, replace=False)
]
x_tmp = U[indexes_causal, :]
x_tmp = np.abs(x_tmp)
x_tmp=x_tmp.flatten()
x_tmp[indexes_causal] = 0
ind_tmp = np.where(x_tmp > 0.5)

IndexError: index 18 is out of bounds for axis 0 with size 1

In [41]:
indexes_causal = indexes[
    np.random.choice(np.arange(len(indexes)), size=1, replace=False)
]
x_tmp = U[indexes_causal, :]
x_tmp = np.abs(x_tmp)
x_tmp=x_tmp.flatten()
x_tmp[indexes_causal] = 0
ind_tmp = np.where(x_tmp > 0.1)
ind_tmp=ind_tmp[0]

In [43]:
indexes_noise = ind_tmp[
    np.random.choice(np.arange(len(ind_tmp)), size=2, replace=False)
]

In [45]:
GWASz[indexes_noise] = GWASz[indexes_noise] + np.random.normal(
    loc=0, scale=scale_noise, size=len(indexes_noise)
)

24/05/17 00:20:21 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 904037 ms exceeds timeout 120000 ms
24/05/17 00:20:21 WARN SparkContext: Killing executors is not supported by current scheduler.
24/05/17 00:20:25 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:301)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:117)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:116)
	at org.apache.spark.storage.B

In [8]:
n_causal = 1
SimSumStatFromLD(n_causal=n_causal,he2_reggen=0.002,U=ld_matrix_for_sim,n=100_000,noise=True,scale_noise=1)

IndexError: index 193 is out of bounds for axis 0 with size 1

In [ ]:
pd.DataFrame.iteritems = pd.DataFrame.items
n_iter=100
ld_index_pd=ld_index.toPandas()
n_caus=2
counter=1
ld_matrix_for_sim=ld_matrix[0:1000,:][:,0:1000]
ld_index_pd=ld_index_pd.iloc[0:1000,:]
cred_sets=None
iteration=0
column_list=['credibleSetIndex', 'studyLocusId', 'studyId', 'region', 'exploded_locus',
       'variantId', 'chromosome', 'position', 'finemappingMethod',
       'credibleSetlog10BF', 'purityMeanR2', 'purityMinR2', 'zScore',
       'pValueMantissa', 'pValueExponent',"is_in_X","is_in_lead"]
for iteration in range(n_iter):
    
    x_cycle=SimSumStat(n_causal=n_caus,he2_reggen=0.002,n=100_000,U=ld_matrix_for_sim)

    if sum(x_cycle["P"]<=5e-8)>0:
        df = pd.DataFrame(
            {
                "z": x_cycle["Z"],
                "variantId": ld_index_pd["variantId"]
            }
        )
        schema = StructType(
            [
                StructField("z", DoubleType(), True),
                StructField("variantId", StringType(), True),
            ]
        )
        df_spark = session.spark.createDataFrame(df, schema=schema)
        
        j=""
        for ii in ld_index_pd["variantId"][x_cycle["indexes"]].tolist():
            j=j+str(ii)+"," 

        CS_sim=SusieFineMapperStep.susie_finemapper_from_prepared_dataframes(
            GWAS_df=df_spark,
            ld_index=ld_index,
            gnomad_ld=ld_matrix,
            L=10,
            session=session,
            studyId="sim"+str(iteration),
            region=j,
            susie_est_tausq=False,
            run_carma= False,
            run_sumstat_imputation = False,
            carma_time_limit = 600,
            imputed_r2_threshold = 0.8,
            ld_score_threshold = 4,
            sum_pips = 0.99,
            primary_signal_pval_threshold = 1e-2,
            secondary_signal_pval_threshold = 1e-2,
            purity_mean_r2_threshold = 0,
            purity_min_r2_threshold = 0,
            cs_lbf_thr =2,
        )
        cred_set=CS_sim["study_locus"].df
        
        # Assuming X is your list
        X = ld_index_pd["variantId"][x_cycle["indexes"]].tolist()

        # Explode the array column
        cred_set = cred_set.withColumn("exploded_locus", col("locus.variantId"))
        # Create a condition for each element in X
        conditions = [array_contains(col("exploded_locus"), x) for x in X]
        # Combine the conditions using the | operator
        combined_condition = conditions[0]
        for condition in conditions[1:]:
            combined_condition = combined_condition | condition
        # Create a new column that is True if any condition is True and False otherwise
        cred_set = cred_set.withColumn("is_in_X", combined_condition)

        cred_set = cred_set.withColumn("is_in_lead", when(col("variantId").isin(X), 1).otherwise(0))

        cred_set=cred_set.toPandas()
        cred_set=cred_set[column_list]
        
        if counter == 1:
            cred_sets = cred_set
        else:
            #cred_sets = cred_sets.unionByName(cred_set)
            cred_sets=pd.concat([cred_sets, cred_set], axis=0)
            #cred_sets=cred_sets.merge(cred_set)
        counter=counter+1
    

In [ ]:
cred_sets1=cred_sets[cred_sets["pValueExponent"]<=-6]


In [ ]:
cred_sets1

In [ ]:
sum(cred_sets1["is_in_X"])/len(cred_sets1["is_in_X"])